In [ ]:
import os

print(os.getcwd())
print(os.listdir())

In [ ]:
os.getcwd()

In [ ]:
import os
print(os.listdir())

In [ ]:
import pandas as pd
import smtplib
from email.message import EmailMessage

# ==========================================
# SETTINGS
# ==========================================

FILE_NAME = "Calculus_Grades_AY2526_2nd_sem.xlsx"  
SHEET_NAME = "Main"

SENDER_EMAIL = "[insert your email here]"
APP_PASSWORD = "[insert your app password here]"

TEST_MODE = False
TEST_EMAIL = "[insert your email here]"

# ==========================================
# LOAD DATA
# ==========================================

df = pd.read_excel("Calculus_Grades_AY2526_2nd_sem.xlsx", sheet_name="Main")

# Keep only rows with email addresses
df = df[df["Email Address:"].astype(str).str.contains("@", na=False)]

# ==========================================
# HELPER FUNCTION
# ==========================================

def format_percent(x):
    if pd.isna(x):
        return ""

    if isinstance(x, (int, float)):
        if x <= 1:
            return f"{x*100:.2f}%"
        return f"{x:.2f}%"

    return str(x)

# ==========================================
# SEND EMAILS
# ==========================================

with smtplib.SMTP_SSL("smtp.gmail.com", 465) as smtp:

    smtp.login(SENDER_EMAIL, APP_PASSWORD)

    for _, row in df.iterrows():

        if pd.isna(row["Final Grade"]):
            continue

        first_name = row["First Name"]
        student_email = row["Email Address:"]

        original_grade = format_percent(row["Final Grade"])
        rounded_grade = format_percent(row["Rounded_FG"])
        original_equiv = f"{float(row['Orig_Equiv']):.2f}"
        equivalent = f"{float(row['Equiv']):.2f}"

        # ==========================================
        # Check if the equivalent of rounded grade is lower than the original equivalent
        # ==========================================

        if float(row["Equiv"]) < float(row["Orig_Equiv"]):
            grade_section = f"""
            <p>
            <ul>
            <li>Final Grade: <strong>{original_grade}</strong> rounded to <strong>{rounded_grade}</strong><br></li>
            <li>Original Equivalent: <strong>{original_equiv}</strong><br></li>
            <li><strong>Final Equivalent: {equivalent}</strong></li>
            </ul>
            </p>
            """
        else:
            grade_section = f"""
            <p>
            <ul>
            <li>Final Grade: <strong>{original_grade}</strong><br></li>
            <li>Equivalent: <strong>{equivalent}</strong></li>
            </ul>
            </p>
            """

        le1 = row["LE1"]
        le2 = row["LE2"]
        le3 = row["LE3"]
        le4 = row["LE4"]

        fe = row["FE"]
        hwsw = format_percent(row["HW/SW"])

        # ==========================================
        # EXCUSED EXAM DETECTION
        # ==========================================

        long_exams = {
            "LE1": le1,
            "LE2": le2,
            "LE3": le3,
            "LE4": le4
        }

        excused_exam = None

        for exam, score in long_exams.items():
            if str(score).strip() == "Excused":
                excused_exam = exam
                break

        if excused_exam:

            fe_weight_text = "Final Exam (45%)"

            included_exams = [
                exam
                for exam in ["LE1", "LE2", "LE3", "LE4"]
                if exam != excused_exam
            ]

            formula_terms = " + ".join(
                [f"({exam}÷30)×0.15" for exam in included_exams]
            )

            formula_note = (
                f"Final Grade(%) = [{formula_terms} "
                f"+ (SW/HW)×0.10 + (FE÷60)×0.45]×100"
            )

        else:

            fe_weight_text = "Final Exam (30%)"

            formula_note = (
                "Final Grade(%) = "
                "[(LE1÷30)×0.15 + "
                "(LE2÷30)×0.15 + "
                "(LE3÷30)×0.15 + "
                "(LE4÷30)×0.15 + "
                "(SW/HW)×0.10 + "
                "(FE÷60)×0.30]×100"
            )

        recipient = TEST_EMAIL if TEST_MODE else student_email

        subject = "Calculus Final Grade"

        plain_text_body = f"""
Good day, {first_name},

Please view this email in HTML format.

Best regards,
[insert your name here]
"""

        html_body = f"""
<html>
<body>

<p>Good day, {first_name},</p>

<p>I hope you are doing well.</p>

<p>Your Calculus <strong>final grade</strong> is:</p>

{grade_section}

<p>Below is a breakdown of your final grade:</p>

<p><strong>Long Exams (15% each):</strong> (out of 30 each)</p>

<ul>
<li>LE1: <strong>{le1}</strong></li> 
<li>LE2: <strong>{le2}</strong></li>
<li>LE3: <strong>{le3}</strong></li>
<li>LE4: <strong>{le4}</strong></li>
</ul>

<p><strong>{fe_weight_text}:</strong> <strong>{fe}</strong> out of 60</p>

<p><strong>HW and SW (10%):</strong> <strong>{hwsw}</strong></p>

<p><strong>Note:</strong><br> {formula_note}
</p>

<p>
If you have any questions or clarifications regarding your grade or scores, please email me by June 8, 2026.
</p>

<p>
If you would like to review your final exam paper, you can visit me at <strong> my office on June 8, 2026 (Monday), from 11:00 AM to 2:00 PM</strong>. I will upload your grade in CRS on June 9, 2026. Once uploaded, you cannot request for any changes.
</p>

<p>
Please take note that, as per the institute's policy, you cannot receive a copy of your final exam papers. You may only view it for the purpose of checking for corrections.
</p>

<p>
I wish you all the best in your future subjects. Thank you very much!

</p>

<p>

</p>

<p>
Best regards,<br>
[insert your name here]
</p>

</body>
</html>
"""

        msg = EmailMessage()
        msg["Subject"] = subject
        msg["From"] = SENDER_EMAIL
        msg["To"] = recipient

        msg.set_content(plain_text_body)
        msg.add_alternative(html_body, subtype="html")

        smtp.send_message(msg)

        print(f"Sent to: {recipient} | Actual student email: {student_email} | Student: {first_name}")

print("Done!")